Requirements
- PyTorch
- Kagglehub
- Pandas
- Datasets

In [1]:
from datasets import load_dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import re

/home/ghawkes/Documents/Projects/Chatbot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_extended_dataset_all_lines(d):
        q_a = {}
        first_d = ""

        first_d = d if first_d == "" else first_d
        prev = ""
        responses = re.split(" '[ \n]*' ", d)
        print("All responses are are ")
        print(responses)
        i = 0
        for resp in responses:
            resp = re.sub(r"[\[\]']", "", resp)
            if i == 0:
                print("Printing first response")
                print(resp)
                print(type(resp))
                print(len(responses))

            
            if i == 0:
                prev = resp
                i = i + 1
                continue
            
            # if i != len(words) - 1:
            if True:
                q_a.update({prev : resp})
                prev = resp
                print("Q_A is")
                print(q_a)
            

            i = i + 1


# Get the first two statements in the form of a dictionary
def get_two_resp(convo):
        q_a = {}
        responses = re.split(" '[ \n]*' ", convo)

        if len(responses) >= 2:
            return responses[0], responses[1]
        else:
             return None, None
        
def clean_str(str):
     str = re.sub("[^ a-zA-Z]+", "", str)
     str = re.sub("[ ]+", " ", str)
     str = str.strip()
     str = str.lower()
     str = re.sub(" [ ]+", " ", str)
     return str
            

def clean(df):
    dialog = df["dialog"]
    
    q_a = {}

    for convo in dialog:
        first_q, first_a = get_two_resp(convo)
        if first_q == None:
              continue

        clean_q = clean_str(first_q)
        clean_a = clean_str(first_a) 

        if len(clean_q) > 0 and len(clean_a) > 0:
            q_a.update({clean_q : clean_a})
    
    print("Questions only: ")
    print(q_a.keys())
    print("Answers only: ")
    print(q_a.values())

    return q_a
    
    # questions= pd.DataFrame(data=q_a)

    # print(questions)

    # return questions



In [3]:
# Set the path to the file you'd like to load
file_path = "test.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "thedevastator/dailydialog-unlock-the-conversation-potential-in",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("Head: ")
print(df.head())
print(df.info())



/tmp/ipykernel_869/3481092370.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Head: 
                                              dialog  \
0  ['Hey man , you wanna buy some weed ? ' ' Some...   
1  ['The taxi drivers are on strike again . ' ' W...   
2  ["We've managed to reduce our energy consumpti...   
3  ['Believe it or not , tea is the most popular ...   
4  ['What are your personal weaknesses ? '\n ' I ...   

                             act                        emotion  
0      [3 2 3 4 3 4 3 2 3 4 2 3]      [0 6 0 0 0 0 0 0 0 0 3 0]  
1                      [1 2 1 1]                      [0 0 0 0]  
2                [1 2 1 2 1 2 1]                [0 0 0 0 0 0 0]  
3  [1 1 1 1 2 2 2 2 1 1 1 3 4 3]  [0 0 0 0 0 0 0 0 0 4 0 0 4 4]  
4              [2 1 2 1 2 1 2 1]              [0 0 0 0 0 0 0 4]  
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   dialog   1000 non-null   str  
 1   act      1000 non-null   str  
 2   emotion  1000 non

In [4]:
dict_qa = clean(df)
print("\nResult:",dict_qa)

Questions only: 
dict_keys(['hey man you wanna buy some weed', 'the taxi drivers are on strike again', 'weve managed to reduce our energy consumption in our factory by about per cent in the last two years thats excellent how have you managed that mainly because weve invested in a heat recovery system what does that mean exactly', 'believe it or not tea is the most popular beverage in the world after water', 'what are your personal weaknesses', 'how long will it take us to drive to london', 'so how did i do on my driving test', 'good morning whats the matter with you good morning doctor i have a terrible headache', 'my dear whats for supper red cooked carp and rape with fresh mushrooms', 'hello this is mike kara', 'sunset hotel may i help you yes i have booked a room for th its a double room hold on please let me check it for you yes youre right you will keep it for days well now i want to change the date from th to th', 'nani book store how can i help you do you have the the man and th

Tokenize the data

In [5]:
import numpy as np

In [6]:
def load_token_dataset(text):
    # Check for capital letters
    if bool(re.search(r'[A-Z]', text)):
        raise ValueError("Text must be all lowercase")
    if bool(re.search(r'[^a-z ]', text)):
        raise ValueError("Text must be all lowercase letters and spaces")
    if bool(re.search(r'  ', text)):
        raise ValueError("Text must not include double spaces")
    if text[0] == " " or text[len(text) - 1] == " ":
        raise ValueError("Text must be stripped")

    token_strs = text.lower().split(" ")
    start_token = 0
    end_token = 1
    token_id = 3
    token_map = {"PAD":0, "START_TOKEN":1, "END_TOKEN":2} # Map token to id quickly
    id_map = ["PAD", "START_TOKEN", "END_TOKEN"] # Index is the token id. Maps id to string token
    for t in token_strs:
        if token_map.get(t) == None:
            token_map[t] = token_id
            id_map.append(t)
            token_id = token_id + 1
    
    return token_map, id_map

def tokenize(text, token_map):
    token_strs = text.lower().split(" ")
    tokens = [token_map["START_TOKEN"]]
    for t in token_strs:
        if token_map.get(t) != None:
            tokens.append(token_map.get(t))
        else:
            pass # Skip unknown inputs
    
    tokens.append(token_map["END_TOKEN"])
    
    return tokens


In [7]:
questions = dict_qa.keys()
answers = dict_qa.values()


question_str = " ".join(questions).strip()
ans_str = " ".join(answers).strip()
full_text = question_str + " " + ans_str

token_map, id_map = load_token_dataset(full_text)

tokenized_inputs = [tokenize(x, token_map) for x in questions]
tokenized_responses = [tokenize(y, token_map) for y in answers]

Filter sentences

In [8]:
MAX_TOKENS = 100

Remove all sentences larger than MAX_TOKENS

In [9]:
removals = []
for i in range(len(tokenized_inputs)):
    if len(tokenized_inputs[i]) > MAX_TOKENS or len(tokenized_responses[i]) > MAX_TOKENS:
        longer = tokenized_inputs[i] if len(tokenized_inputs[i]) > MAX_TOKENS else tokenized_responses[i]
        print(longer)
        print(len(longer))
        removals.append(i)

for i in reversed(removals):
    tokenized_inputs.pop(i)
    tokenized_responses.pop(i)


[1, 77, 78, 365, 366, 217, 61, 170, 95, 211, 367, 27, 333, 41, 368, 369, 370, 81, 41, 371, 307, 79, 14, 96, 372, 118, 140, 65, 151, 373, 374, 14, 375, 376, 91, 377, 140, 65, 151, 41, 378, 379, 26, 221, 370, 380, 14, 381, 376, 107, 41, 77, 382, 37, 72, 36, 383, 181, 127, 5, 182, 112, 27, 10, 132, 14, 375, 384, 107, 19, 385, 255, 86, 10, 386, 45, 27, 10, 374, 387, 388, 107, 41, 389, 370, 390, 10, 391, 392, 393, 24, 394, 395, 41, 396, 27, 10, 397, 391, 398, 107, 41, 399, 91, 400, 401, 50, 402, 54, 91, 10, 403, 54, 186, 10, 404, 2]
117
[1, 139, 414, 211, 354, 19, 36, 41, 415, 416, 196, 417, 142, 5, 418, 112, 8, 419, 384, 420, 136, 352, 50, 263, 315, 421, 86, 5, 52, 19, 151, 422, 415, 423, 424, 425, 426, 313, 315, 427, 136, 41, 428, 429, 430, 431, 432, 59, 10, 433, 434, 157, 19, 435, 37, 436, 142, 52, 437, 10, 438, 51, 37, 140, 362, 439, 440, 210, 33, 315, 421, 70, 5, 441, 442, 443, 10, 438, 10, 444, 445, 91, 217, 10, 446, 434, 216, 447, 19, 207, 45, 10, 448, 13, 449, 70, 10, 196, 444, 5, 6

Pad all sentences

In [10]:
def pad_inputs(tokenized_inputs, tokenized_responses, token_map, max_tokens):
    for i in range(len(tokenized_inputs)):
        if len(tokenized_inputs[i]) < max_tokens:
            tokenized_inputs[i] = tokenized_inputs[i] + [token_map["PAD"]]*(max_tokens - len(tokenized_inputs[i]))
        if len(tokenized_responses[i]) < max_tokens:
            tokenized_responses[i] = tokenized_responses[i] + [token_map["PAD"]]*(max_tokens - len(tokenized_responses[i]))

In [11]:
pad_inputs(tokenized_inputs, tokenized_responses, token_map, MAX_TOKENS)

Positional encoding - gives the model the original order of inputs

The next block uses code from 

In [ ]:
def get_angles(pos, i, d_model):
    angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
    return pos * angle_rates

def positional_encoding(position, d_model):
    angle_rads = get_angles(np.arange(position)[:, np.newaxis], np.arange(d_model)[np.newaxis, :], d_model)
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2]) #for even positions using sin()
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2]) #for odd positions using cos()
    pos_encoding = angle_rads[np.newaxis,:]
    return pos_encoding